# Train Point-SAM on real forest crops

Fine-tunes Point-SAM on the ISPRS Wytham **train** strip and picks the checkpoint on the **val** strip. The **test** strip is not used here: the benchmark notebook scores it once, at the end.

**Needs a CUDA GPU.** It runs top to bottom, either interactively in Jupyter on a GPU node, or unattended:
`sbatch scripts/slurm/run_notebook.sbatch notebooks/02_train.ipynb configs/experiment/isprs_lora.yaml`

Everything logged also goes to `history.csv` as it happens, so if a session drops, the last section can still draw the curves.

## 1. Setup and config

The config is the record of the run, so settings are changed there and not in this notebook.

In [ ]:
import os
import sys

if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, "src")

import numpy as np
import torch
import yaml
from IPython.display import Image, display
from torch.utils.data import Subset

from treeseg.data.dataset import ForestCrops
from treeseg.engine.trainer import train
from treeseg.models.point_sam import add_point_sam_to_path, build_point_sam, prepare_for_lora
from treeseg.utils.curves import make_all

config_path = os.environ.get("TREESEG_CONFIG", "configs/experiment/smoke.yaml")
cfg = yaml.safe_load(open(config_path))
add_point_sam_to_path()
print("config:", config_path)
print("device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NO GPU: this notebook will fail at training")

## 2. Data

Ground is dropped from validation because inference removes it too. Clicks during validation come from the true labels, so these numbers measure masks, not automatic clicking.

In [ ]:
train_set = ForestCrops(os.path.join(cfg["data_dir"], "train"), cfg["num_points"], cfg["num_masks"],
                        cfg["min_mask_points"], augmentation=cfg["augmentation"], seed=cfg["seed"])
val_full = ForestCrops(os.path.join(cfg["data_dir"], "val"), cfg["num_points"], cfg["val_num_masks"],
                       cfg["min_mask_points"], drop_ground=True, seed=cfg["seed"] + 1)
val_set = Subset(val_full, range(min(cfg["val_max_crops"], len(val_full))))

steps_per_epoch = len(train_set) / (cfg["batch_size"] * cfg["grad_accum"])
print(f"train crops: {len(train_set)}, val crops: {len(val_set)}")
print(f"{cfg['max_steps']} steps is about {cfg['max_steps'] / steps_per_epoch:.1f} epochs")

item = train_set[0]
print({key: tuple(value.shape) for key, value in item.items()})
print("share of points in each target mask:", item["gt_masks"].float().mean(dim=1).round(decimals=3).tolist())

## 3. Model

Stock Point-SAM with LoRA adapters. Only the adapters, patch embedding and decoders train.

In [ ]:
model = build_point_sam(cfg["checkpoint"], cfg["group_number"], cfg["group_size"])
prepare_for_lora(model, cfg["lora_rank"], cfg["lora_alpha"])

## 4. Train

Saves `best.safetensors` whenever `val_iou_pick` improves, and stops early after 6 validations without improvement.

In [ ]:
torch.cuda.reset_peak_memory_stats()
history_path = train(model, train_set, val_set, cfg, cfg["out_dir"])
print(f"peak GPU memory: {torch.cuda.max_memory_allocated() / 1e9:.1f} GB")

## 5. Curves

This section only needs `history.csv`, so it also works on a partial run. If the session dropped, set `history_path` to `<out_dir>/history.csv` and run the imports in section 1 first.

In [ ]:
history_path = os.path.join(cfg["out_dir"], "history.csv")
paths = make_all(history_path, os.path.join(cfg["out_dir"], "figures"))
for name in ("training_loss", "validation_iou", "epoch_loss"):
    display(Image(paths[name]))

In [ ]:
import pandas as pd

history = pd.read_csv(history_path)
val = history[history["split"] == "val"]
best = val.loc[val[cfg["select_metric"]].idxmax()]
print(f"best {cfg['select_metric']} = {best[cfg['select_metric']]:.3f} at step {int(best['step'])} (epoch {best['epoch']:.1f})")
print(f"ran for {history['minutes'].max():.1f} minutes")
pd.read_csv(paths["per_epoch"]).round(4)